# Capstone Project 3: Dialogue Summarization (SAMSum)

## Problem Statement and Approach
Information overload in very active group chats reduces user engagement and makes it harder to catch up after being away. A summary feature can condense long dialogue threads into short, accurate summaries that capture the key points in the conversation. In this project, we use the SAMSum dataset, dialogues paired with summaries, to build a prototype that converts a dialogue into a brief summary.  

## Problem Description
The input is a multi-turn dialogue from a group chat.  The output will be a concise summary that preserves the important facts and agreements and other key points.  The goal is to demonstrate that an encoder–decoder transformer can reliably generate summaries suitable for a user-facing product feature.  

## Approach  
The project will follow an end-to-end NLP workflow: data exploration, then work on preprocessing and tokenization, then model design (BERT-based encoder–decoder), followed by fine tuning and evaluation with ROUGE, finally we will assess our error analysis and make improvements.  


## Import Libraries and Data

In [ ]:
# Environment Setup

import os 
import re 
import math 
import json 
import random 
from dataclasses import dataclass 
from pathlib import Path

import numpy as np 
import pandas as pd

import torch

import matplotlib.pyplot as plt 
import seaborn as sns

from IPython.display import display

# Transformers
from transformers import ( AutoTokenizer, EncoderDecoderModel, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback, ) 
from transformers import set_seed

# Metrics (ROUGE)
import evaluate

# Paths (Subfolder is Data, the three files are train.csv, validation.csv, and test.csv)

PROJECT_DIR = Path.cwd() 
DATA_DIR = PROJECT_DIR / "Data"

TRAIN_PATH = DATA_DIR / "train.csv" 
VAL_PATH = DATA_DIR / "validation.csv" 
TEST_PATH = DATA_DIR / "test.csv"

assert TRAIN_PATH.exists(), f"Missing: {TRAIN_PATH}" 
assert VAL_PATH.exists(), f"Missing: {VAL_PATH}" 
assert TEST_PATH.exists(), f"Missing: {TEST_PATH}"

# Reproducibility
SEED = 42 
set_seed(SEED) 
random.seed(SEED) 
np.random.seed(SEED) 
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu" 
print("DEVICE:", DEVICE)

# Tokenization / sequence length params (you will tune after EDA)

ENC_MAX_LEN = 256 # encoder (dialogue) max length 
DEC_MAX_LEN = 64 # decoder (summary) max length

# Generation params (baseline)
GEN_NUM_BEAMS = 4 
GEN_LENGTH_PENALTY = 1.0 
GEN_MAX_LENGTH = DEC_MAX_LEN 
GEN_NO_REPEAT_NGRAM_SIZE = 3

In [ ]:
# Load data & basic validation 
train_df = pd.read_csv(TRAIN_PATH) 
val_df = pd.read_csv(VAL_PATH) 
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape) 
print("Columns:", train_df.columns.tolist())

# Basic missing values check
print("\nMissing values (train):") 
print(train_df.isna().sum())

# Basic sample preview
print("\nSample train row:") 
display(train_df.head(1))


In [ ]:
# EDA: word-length distributions, summary ratio, sanity checks
def word_count(x: str) -> int: 
    x = "" if pd.isna(x) else str(x) 
    x = x.replace("\n", " ") 
    toks = [t for t in x.split(" ") if t.strip()] 
    return len(toks)

# Dialogue word count & summary word count
train_df["dialogue_words"] = train_df["dialogue"].astype(str).apply(word_count) 
train_df["summary_words"] = train_df["summary"].astype(str).apply(word_count) 
train_df["ratio"] = train_df["summary_words"] / train_df["dialogue_words"].replace(0, np.nan)

val_df["dialogue_words"] = val_df["dialogue"].astype(str).apply(word_count) 
val_df["summary_words"] = val_df["summary"].astype(str).apply(word_count) 
val_df["ratio"] = val_df["summary_words"] / val_df["dialogue_words"].replace(0, np.nan)

print("Train dialogue word count summary:") 
print(train_df["dialogue_words"].describe()) 
print("\nTrain summary word count summary:") 
print(train_df["summary_words"].describe())

# Plot distributions
sns.set(style="whitegrid")

plt.figure(figsize=(12,4)) 
sns.histplot(train_df["dialogue_words"], bins=60, kde=False) 
plt.title("Train: Dialogue word count distribution") 
plt.show()

plt.figure(figsize=(12,4)) 
sns.histplot(train_df["summary_words"], bins=60, kde=False, color="orange") 
plt.title("Train: Summary word count distribution") 
plt.show()

plt.figure(figsize=(12,4)) 
sns.histplot(train_df["ratio"].dropna(), bins=50, kde=False, color="green") 
plt.title("Train: Summary/Dialogue word ratio distribution") 
plt.show()

# Show a couple examples with extremely long dialogues or short summaries (helps truncation decisions)
print("\nLongest dialogues (train):") 
display(train_df.sort_values("dialogue_words", ascending=False).head(3)[["id","dialogue","summary","dialogue_words","summary_words"]])

print("\nShortest summaries (train):") 
display(train_df.sort_values("summary_words", ascending=True).head(3)[["id","dialogue","summary","dialogue_words","summary_words"]])

## The following section is designed to normalize speaker markers to reduce token noise

1) Replace "Name:" / "A:" / "B:" speaker prefixes with a generic "SPEAKER:"
2) Replace any <...> markup tokens (e.g. <file_gif>) with a stable token "[MARKUP]"
3) Normalize whitespace

In [ ]:
# Preprocessing: normalize speakers & replace noisy markup tokens

SPEAKER_PREFIX_RE = re.compile(r"(?m)^\s*([A-Za-z][A-Za-z0-9_-])\s:\s*") # start-of-line "X:" (common speaker label pattern) 
MARKUP_RE = re.compile(r"<[^>]+>") # anything like <file_gif>

def normalize_dialogue(text: str) -> str: 
    if pd.isna(text): 
        return "" 
    text = str(text)

# Replace speaker labels with generic token
    text = SPEAKER_PREFIX_RE.sub("SPEAKER: ", text)

# Replace markup tokens with a stable placeholder
    text = MARKUP_RE.sub("[MARKUP]", text)

# Normalize whitespace
    text = text.replace("\r", "\n")
    text = text.replace("\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s+", "\n", text)
    text = text.strip()

    return text

In [ ]:
def normalize_summary(text: str) -> str: 
        if pd.isna(text): 
            return "" 
        text = str(text) 
        text = text.replace("\r", " ") 
        text = re.sub(r"\s+", " ", text).strip() 
        return text

train_df["dialogue_clean"] = train_df["dialogue"].astype(str).apply(normalize_dialogue) 
train_df["summary_clean"] = train_df["summary"].astype(str).apply(normalize_summary)

val_df["dialogue_clean"] = val_df["dialogue"].astype(str).apply(normalize_dialogue) 
val_df["summary_clean"] = val_df["summary"].astype(str).apply(normalize_summary)

test_df["dialogue_clean"] = test_df["dialogue"].astype(str).apply(normalize_dialogue) 
test_df["summary_clean"] = test_df["summary"].astype(str).apply(normalize_summary)

print("Before/after example (dialogue):") 
ex_i = 0 
display(train_df.loc[ex_i, ["dialogue", "dialogue_clean"]])

print("\nBefore/after example (summary):") 
display(train_df.loc[ex_i, ["summary", "summary_clean"]])

In [ ]:
# Token-length stats require a tokenizer
print("\nLoading tokenizer for token-length analysis...")

tokenizer_name = "bert-base-uncased" 
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

def count_tokens(text: str) -> int: # We only want length estimates; no truncation/padding here. 
    return len(tokenizer.encode(text, add_special_tokens=True))

train_df["dialogue_tokens"] = train_df["dialogue_clean"].apply(count_tokens) 
train_df["summary_tokens"] = train_df["summary_clean"].apply(count_tokens)

print("\nToken count stats (train):") 
print("Dialogue tokens:", train_df["dialogue_tokens"].describe()) 
print("Summary tokens :", train_df["summary_tokens"].describe())

In [ ]:
# Max lengths by percentile coverage
for p in [90, 95, 97, 99]: 
    print(f"Dialogue token {p}th percentile:", np.percentile(train_df["dialogue_tokens"], p)) 
    print(f"Summary token {p}th percentile:", np.percentile(train_df["summary_tokens"], p)) 
    print("----")

plt.figure(figsize=(12,4)) 
sns.histplot(train_df["dialogue_tokens"], bins=80, kde=False) 
plt.title("Train: Dialogue token count distribution (approx)") 
plt.show()

plt.figure(figsize=(12,4)) 
sns.histplot(train_df["summary_tokens"], bins=80, kde=False, color="orange") 
plt.title("Train: Summary token count distribution (approx)") 
plt.show()

print("Suggested ENC_MAX_LEN:", int(np.percentile(train_df["dialogue_tokens"], 95))) 
print("Suggested DEC_MAX_LEN:", int(np.percentile(train_df["summary_tokens"], 95)))

In [ ]:
# Lead-k extractive baseline & ROUGE evaluation
# In the pitch for this project, these were the tools referenced

rouge = evaluate.load("rouge")

def split_into_sentences(dialogue: str): # Simple sentence segmentation. (Dataset lines may use speaker turns; still helps.) 
    # If a dialogue is multiline, this treats punctuation as boundaries. 
    dialogue = dialogue.replace("\n", " ").strip() 
    parts = re.split(r'(?<=[.!?])\s+', dialogue) 
    parts = [p.strip() for p in parts if p.strip()] 
    return parts

def lead_k_summary(dialogue_clean: str, k: int = 3) -> str: 
    sents = split_into_sentences(dialogue_clean) 
    chosen = sents[:k] 
    if len(chosen) == 0: 
        return "" 
    return " ".join(chosen)

def rouge_eval_texts(pred_texts, ref_texts): 
    # Use ROUGE F1 (evaluate returns precision/recall/fmeasure) 
    # We'll compute with stemming enabled for stability. 
    results = rouge.compute(predictions=pred_texts, references=ref_texts, use_stemmer=True) 
    # results keys typically: rouge1, rouge2, rougeL 
    return results

def evaluate_lead_k_on_df(df, k: int = 3): 
    preds = [lead_k_summary(d, k=k) for d in df["dialogue_clean"].tolist()] 
    refs = df["summary_clean"].tolist() 
    results = rouge_eval_texts(preds, refs) 
    return results, preds

In [ ]:
# Evaluate baseline on validation
LEAD_K = 3
baseline_results, baseline_preds = evaluate_lead_k_on_df(val_df, k=LEAD_K)

print(f"Lead-{LEAD_K} baseline ROUGE on validation:") 
print(baseline_results)

# Create a small qualitative table
print("\nQualitative baseline examples (val):") 
show_n = 5 
idxs = np.random.choice(len(val_df), 
                       size=min(show_n, len(val_df)), 
                       replace=False)

rows = []
for i in idxs:
    # Check if baseline_preds is a list/array or a single value
    if isinstance(baseline_preds, (list, np.ndarray)) and len(baseline_preds) > i:
        pred_text = baseline_preds[i][:200]
    else:
        # If baseline_preds is a scalar or doesn't have enough elements, use it directly
        pred_text = str(baseline_preds)[:200]
    
    rows.append({
        "id": val_df.iloc[i]["id"],
        "dialogue_sample": str(val_df.iloc[i]["dialogue_clean"])[:120] + "...",
        "pred_lead_k": pred_text,  # Fixed: handle both scalar and array cases
        "reference_summary": val_df.iloc[i]["summary_clean"][:200],
    })

display(pd.DataFrame(rows))

# Plot ROUGE (baseline single-point bar chart)
# labels = ["rouge1", "rouge2", "rougeL"] 
# f1s = [baseline_results[l]["fmeasure"] for l in labels] 
    
# plt.figure(figsize=(6,4)) 
# sns.barplot(x=labels, y=f1s) 
# plt.title(f"Lead-{LEAD_K} baseline ROUGE F1") 
# plt.ylabel("F1") 
# plt.show()

In [ ]:
# HuggingFace's BERT encoder/decoder
from transformers.utils import logging as hf_logging
import warnings

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", category=UserWarning, message=r".*as_target_tokenizer is deprecated.*")

model = EncoderDecoderModel.from_encoder_decoder_pretrained( tokenizer_name, tokenizer_name )

# For BERT:
# CLS as decoder start
# SEP as EOS (common choice for BERT-family)
model.config.decoder_start_token_id = tokenizer.cls_token_id 
model.config.eos_token_id = tokenizer.sep_token_id 
model.config.pad_token_id = tokenizer.pad_token_id

model.to(DEVICE)

# Tokenization for seq2seq
def preprocess_function(examples): 
    # Encoder inputs 
    model_inputs = tokenizer( 
        examples["dialogue_clean"], 
        max_length=ENC_MAX_LEN, 
        padding="max_length", 
        truncation=True 
    )

    # Decoder labels - Fixed indentation to be inside the function
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            examples["summary_clean"],
            max_length=DEC_MAX_LEN,
            padding="max_length",
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# HuggingFace dataset-like formatting

class SimpleDataset(torch.utils.data.Dataset): 
    def __init__(self, df):  
        self.df = df.reset_index(drop=True)

        # Pre-tokenize once (faster training later) 
        enc = preprocess_function({
            "dialogue_clean": self.df["dialogue_clean"].tolist(),
            "summary_clean":  self.df["summary_clean"].tolist()
        })

        self.input_ids = torch.tensor(enc["input_ids"], dtype=torch.long)
        self.attention_mask = torch.tensor(enc["attention_mask"], dtype=torch.long)
        self.labels = torch.tensor(enc["labels"], dtype=torch.long)

        # Replace padding token id in labels with -100 so loss ignores them
        self.labels[self.labels == tokenizer.pad_token_id] = -100

    def __len__(self):  
        return len(self.df)

    def __getitem__(self, idx):  
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

train_dataset = SimpleDataset(train_df) 
val_dataset = SimpleDataset(val_df)

print("Train dataset size:", len(train_dataset), "Val dataset size:", len(val_dataset))

    


In [ ]:
# Metrics for Trainer

def decode_batch_ids(token_ids_batch): # token_ids_batch: numpy array or list # Replace -100 with pad for decoding 
    token_ids_batch = np.array(token_ids_batch) 
    token_ids_batch = np.where(token_ids_batch == -100, tokenizer.pad_token_id, token_ids_batch) 
    texts = tokenizer.batch_decode(token_ids_batch, skip_special_tokens=True, clean_up_tokenization_spaces=True) # Strip extra whitespace 
    texts = [t.strip() for t in texts] 
    return texts

def postprocess_generation(texts): # Optional: you can do extra cleanup here 
    return [t.strip() for t in texts]

def compute_metrics(eval_pred):
    # Extract predictions and labels from the EvalPrediction object
    preds, labels = eval_pred.predictions, eval_pred.label_ids
    
    # Decode predictions - use preds instead of undefined pred_ids
    texts = tokenizer.batch_decode(preds, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    
    # Process the decoded texts - use texts instead of undefined pred_texts
    pred_texts = postprocess_generation([t.strip() for t in texts])
    label_texts = decode_batch_ids(labels)

    results = rouge.compute(
        predictions=pred_texts,
        references=label_texts,
        use_stemmer=True
    )
    # Trainer expects a flat dict of floats for logging
    return {
        "rouge1_f": results["rouge1"].mid.fmeasure,
        "rouge2_f": results["rouge2"].mid.fmeasure,
        "rougeL_f": results["rougeL"].mid.fmeasure,
    }

# Training args (with early stopping/checkpointing)
output_dir = str(PROJECT_DIR / "outputs_seq2seq_bert_encoder_decoder")

training_args = Seq2SeqTrainingArguments( 
    output_dir=output_dir, overwrite_output_dir=True,
    eval_strategy="epoch",
    save_strategy="epoch",

# Training hyperparams — choose conservative defaults; tune later 
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,

    learning_rate=5e-5,
    num_train_epochs=7,
    weight_decay=0.01,

    warmup_steps=0,
    lr_scheduler_type="linear",

    fp16=torch.cuda.is_available(),  # mixed precision if GPU

    predict_with_generate=True,
    generation_max_length=GEN_MAX_LENGTH,

    logging_steps=25,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="rougeL_f",
    greater_is_better=True,

    seed=SEED,
    dataloader_num_workers=0,
    # Fixed: Set to True to remove incompatible parameters
    remove_unused_columns=True,  # This filters out unexpected parameters
    # Add this to handle the specific compatibility issue
    dataloader_drop_last=False,
)

# Updated data collator with minimal parameters to avoid conflicts
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=model,
    padding=True,
    return_tensors="pt",
    # Removed max_length parameter as it can cause conflicts
)

early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

trainer = Seq2SeqTrainer( 
    model=model, 
    args=training_args, 
    train_dataset=train_dataset, 
    eval_dataset=val_dataset, 
    data_collator=data_collator, 
    tokenizer=tokenizer, 
    compute_metrics=compute_metrics,
    callbacks=[early_stopping], 
)

print("Starting training...")  
train_result = trainer.train()  
print("Training finished.")

print("\nBest metric (Trainer):") 
print(trainer.state.best_metric)